# Face-Car — treinamento CNN no Google Colab

Este notebook treina somente CNNs com imagens reais rotuladas do YawDD. As classes `bocejo` e `sem_bocejo` representam sinais visuais, não uma classificação completa de sonolência.

O notebook original do professor, `Aula10_Colab_CIFAR10.ipynb`, é preservado como referência.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q tensorflow opencv-python-headless scikit-learn matplotlib

import os, sys, subprocess
from pathlib import Path
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU disponível:', bool(tf.config.list_physical_devices('GPU')))

## Caminhos no Google Drive

Coloque o manifesto supervisionado e os vídeos do YawDD no Drive. O notebook não baixa o dataset automaticamente.

In [ ]:
DATASET_DRIVE = Path('/content/drive/MyDrive/Face-Car/datasets')
DATA_DIR = DATASET_DRIVE / 'yawdd_processado'
MANIFESTO = DATASET_DRIVE / 'manifesto_yawdd.csv'
RESULTADOS = Path('/content/drive/MyDrive/Face-Car/resultados_cnn')
RESULTADOS.mkdir(parents=True, exist_ok=True)
print('Dataset:', DATA_DIR)
print('Manifesto:', MANIFESTO)
print('Resultados:', RESULTADOS)

## Preparar frames

O manifesto deve conter `video_path,label,driver_id`. Os rótulos precisam vir de anotação supervisionada; o script não infere rótulos por nome de arquivo. A divisão por motorista/vídeo evita vazamento entre treino, validação e teste.

In [ ]:
REPO = Path('/content/face-car')
if not (REPO / 'treinamento' / 'preparar_dados.py').exists():
    subprocess.run(['git', 'clone', '--branch', 'feature/treinamento-avaliacao', '--single-branch', 'https://github.com/dario-bastos-dev/face-car.git', str(REPO)], check=True)
PREPARAR = REPO / 'treinamento' / 'preparar_dados.py'
if MANIFESTO.exists() and not (DATA_DIR / 'metadata.csv').exists():
    subprocess.run([sys.executable, str(PREPARAR), '--manifest', str(MANIFESTO), '--videos-dir', str(DATASET_DRIVE), '--output-dir', str(DATA_DIR), '--frame-stride', '5', '--image-size', '96'], check=True)
else:
    print('Preparação ignorada: use um manifesto existente e execute quando os vídeos estiverem disponíveis.')

## Treinar os modelos A e B

Modelo A: CNN básica, até 15 épocas. Modelo B: augmentation, Dropout de 50%, EarlyStopping e até 30 épocas.

In [ ]:
TREINAR = REPO / 'treinamento' / 'treinar.py'
subprocess.run([sys.executable, str(TREINAR), '--data-dir', str(DATA_DIR), '--output-dir', str(RESULTADOS), '--image-size', '96', '--batch-size', '64'], check=True)

## Avaliar e salvar o melhor modelo

São calculados acurácia, precisão, recall, F1-score e matrizes de confusão. O melhor modelo é escolhido pela validação e salvo como `melhor_modelo.keras`. Os resultados só existem após treinamento real.

In [ ]:
AVALIAR = REPO / 'treinamento' / 'avaliar.py'
subprocess.run([sys.executable, str(AVALIAR), '--data-dir', str(DATA_DIR), '--models-dir', str(RESULTADOS), '--image-size', '96', '--batch-size', '64'], check=True)
print('Arquivos salvos em:', RESULTADOS)

## Pendências antes do treinamento real

1. Obter o YawDD conforme sua licença.
2. Criar e revisar o manifesto com rótulos `bocejo`/`sem_bocejo` e `driver_id`.
3. Executar a preparação e revisar a distribuição das classes.
4. Treinar com GPU, avaliar e analisar os erros.
5. Validar posteriormente a integração do recorte facial com o MediaPipe do projeto.